# Helios 04 — Distributed Sharding & Mosaic

![Distributed PMTiles sharding diagram](https://raw.githubusercontent.com/databrickslabs/geobrix/main/resources/images/helios-04.png)

**Solar site-selection, step 4: scaling to regions.** Notebook 01 wrote the entire SF
building pyramid into a **single** PMTiles archive — perfect for a city. When the area
grows to statewide or continental scale, a single archive becomes a multi-gigabyte
monolith: slow to re-generate when a neighbourhood updates, and a bottleneck for
parallel workers.

This notebook demonstrates the **distributed sharding** alternative:

1. **Assign every tile to a coarse parent shard** using a bit-shift on `(z, x, y)`.
   Each shard key corresponds to one parent tile in the slippy-map quadtree.
2. **One PMTiles archive per shard** — `gbx_pmtiles_agg` grouped by `shard` produces
   one bounded archive per key instead of one global archive.
3. **Mosaic catalog** — a lightweight `mosaic.json` (and a Delta table) maps each
   shard's bounds and zoom range to its Volume path. A MapLibre + pmtiles.js client
   fetches whichever shard(s) the viewport intersects; the browser assembles the full
   map client-side without any tile-proxy server.

**Key rules from the sharding mental model:**
- **Buffer the source query** so features straddling a boundary are not clipped inside
  either shard's archive. These demo notebooks use a small AOI and omit source buffering;
  a production pipeline must expand the source geometry query by a small buffer so that
  features near a shard boundary are fetched in full before tiling clips them to each
  tile's boundary.
- **Output tiles must not overlap across shards** — the same `(z, x, y)` must appear
  in exactly one archive. The bit-shift assignment guarantees this: every tile's parent
  at `shard_z` is unique.
- **Atomic re-generation** — if a single neighbourhood updates, only the one shard
  that contains it needs to be re-generated; the rest of the catalog is untouched.

> Runs on the **lightweight tier (Serverless)** by default. See `config_nb` for the
> heavyweight switch.

In [ ]:
%run ./config_nb

In [ ]:
# Flip to True to fully rebuild this notebook's tables / re-download / re-tile.
FORCE_REBUILD = False

## 1. Build the tile rows

We load the same SF building footprints as Notebook 01 and run `gbx_st_asmvt_pyramid`
to produce the flat `(z, x, y, mvt_bytes)` tile pyramid. If the Overture network is
unavailable (Docker / CI / `GBX_HELIOS_OFFLINE=1`) we fall back to the same synthetic
SF buildings used in Notebook 01.

We extend the zoom range to **z11–z16** so the MVT rows include tiles at `shard_z=11`
itself (needed so the shard assignment is non-trivial even at the coarsest zoom).

In [ ]:
import json
import os
from pyspark.sql.types import StringType, BinaryType, StructType, StructField

# -- Offline guard (mirrors NB01) ----------------------------------------------
_OFFLINE = os.environ.get("GBX_HELIOS_OFFLINE", "").lower() in ("1", "true", "yes")

if not _OFFLINE:
    try:
        assets = overture.discover(SF_CITY_BBOX, themes=["buildings"])
        print(f"... {assets.count()} intersecting building assets")
    except Exception as _e:
        print(f"... discover failed ({type(_e).__name__}: {_e}); switching to offline mode")
        _OFFLINE = True

if _OFFLINE:
    print("... OFFLINE mode: Overture network unavailable; using synthetic SF buildings fallback")
    assets = None

In [ ]:
OVERTURE_DIR = f"{HELIOS_DIR}/overture"

if not _OFFLINE:
    meta = overture.download(
        assets, OVERTURE_DIR,
        table="overture_buildings_meta",
        validate=True,
        partitions=64,
    )
else:
    meta = None
    print("... OFFLINE mode: skipping download")

In [ ]:
if not _OFFLINE:
    buildings = (
        overture.read("overture_buildings_meta", theme="buildings", type="building", bbox=SF_CITY_BBOX)
                .select(F.col("id").alias("feature_id"), F.col("geometry"))
    )
    print(f"... {buildings.count():,} building footprints in the AOI")
else:
    # Synthetic SF-area building footprints as WKB BINARY (EPSG:4326 polygons).
    #
    # z11 shard boundaries near SF:
    #   lon boundary (x=326|327): -122.5195  (east of this → x=327)
    #   lat boundary (y=791|792): 37.7186   (south of this → y=792)
    #
    # Two footprints per shard, placed well clear of boundaries:
    #
    #   11_326_791  west of -122.5195, north of 37.7186  (Outer Sunset / Inner Sunset)
    #   11_326_792  west of -122.5195, south of 37.7186  (Daly City / Westlake)
    #   11_327_791  east of -122.5195, north of 37.7186  (Mission / Noe Valley)
    #   11_327_792  east of -122.5195, south of 37.7186  (south SF east / Daly City east)
    from shapely.geometry import box as _shapely_box
    from shapely import wkb as _wkb

    _sf_parcels = [
        # (id, minx, miny, maxx, maxy)
        # -- 11_326_791 : Outer Sunset, Inner Sunset --
        ("b001", -122.5505, 37.7600, -122.5495, 37.7610),
        ("b002", -122.5305, 37.7800, -122.5295, 37.7810),
        # -- 11_326_792 : Daly City, Westlake --
        ("b003", -122.5505, 37.6800, -122.5495, 37.6810),
        ("b004", -122.5305, 37.6500, -122.5295, 37.6510),
        # -- 11_327_791 : Mission, Noe Valley --
        ("b005", -122.4405, 37.7800, -122.4395, 37.7810),
        ("b006", -122.4205, 37.7600, -122.4195, 37.7610),
        # -- 11_327_792 : south SF east, Daly City east --
        ("b007", -122.4405, 37.6800, -122.4395, 37.6810),
        ("b008", -122.4105, 37.6500, -122.4095, 37.6510),
    ]

    _schema = StructType([
        StructField("feature_id", StringType(), False),
        StructField("geometry", BinaryType(), True),
    ])
    _rows = [
        (_id, bytes(_wkb.dumps(_shapely_box(minx, miny, maxx, maxy))))
        for _id, minx, miny, maxx, maxy in _sf_parcels
    ]
    buildings = spark.createDataFrame(_rows, schema=_schema)
    print(f"... {buildings.count()} synthetic SF building footprints (offline mode)")
    display(buildings.limit(5))

In [ ]:
buildings.createOrReplaceTempView("sf_buildings_nb04")

# gbx_st_asmvt_pyramid (light tier — Python UDTF):
#   gbx_st_asmvt_pyramid(geom_wkb, attrs, min_z, max_z [, layer_name [, extent]])
# Output columns (flat): z INTEGER, x INTEGER, y INTEGER, mvt_bytes BINARY
#
# Zoom range z11-z16: includes tiles at shard_z=11 itself so the shard assignment
# produces non-trivial results even at the coarsest zoom level.
mvt = spark.sql("""
    SELECT t.z, t.x, t.y, t.mvt_bytes
    FROM sf_buildings_nb04,
         LATERAL gbx_st_asmvt_pyramid(
             geometry,
             named_struct('feature_id', feature_id),
             11, 16,
             'buildings'
         ) AS t
""")

sf_mvt_nb04 = finalize_delta(mvt, "sf_buildings_mvt_shards_src")
print(f"... {sf_mvt_nb04.count():,} (z,x,y) MVT rows across z11-z16")

## 2. Assign each tile to a coarse parent shard

The shard key is derived by right-shifting `x` and `y` to a coarser zoom level
(`shard_z`). For any tile at zoom `z >= shard_z`:

```
sx = x >> (z - shard_z)   # parent tile X at shard_z
sy = y >> (z - shard_z)   # parent tile Y at shard_z
shard = f"{shard_z}_{sx}_{sy}"
```

This is a **lossless spatial partition**: every tile maps to exactly one parent shard
tile, and two tiles that share the same parent will always land in the same shard.
Output tiles are therefore **non-overlapping across shards** by construction.

**`shard_z = 11` for this SF demo.** The SF bounding box spans two z11 tile columns
(x = 326, 327) and two rows (y = 791, 792), yielding **four shards** —
`11_326_791`, `11_326_792`, `11_327_791`, `11_327_792`. Four archives with a city
dataset illustrate the multi-archive pattern without overwhelming the driver.

> At continental scale you would use a coarser `shard_z` (e.g. 6 for ~64 world tiles,
> or 3 for ~8), keeping each archive in the 100 MB–2 GB management sweet spot and
> limiting per-archive tile counts to a processable number.

In [ ]:
# shard_z=11 splits the SF AOI into 4 parent tiles: x∈{326,327}, y∈{791,792}.
# Each tile at zoom z is right-shifted by (z - shard_z) to find its parent.
# Tiles with z < shard_z are excluded here since our pyramid starts at z=11=shard_z.
SHARD_Z = 11

# F.shiftright(col, numBits) requires a Python int literal for numBits, not a Column.
# Since the shift amount is z - SHARD_Z (varies per row) we use F.expr with inline SQL
# arithmetic — shiftright in Spark SQL accepts Column expressions for both operands.
shard_key = F.expr(
    f"concat('{SHARD_Z}_', shiftright(x, z - {SHARD_Z}), '_', shiftright(y, z - {SHARD_Z}))"
)

mvt_sharded = sf_mvt_nb04.withColumn("shard", shard_key)

# Count unique shards to confirm we get >1
shard_counts = (
    mvt_sharded.groupBy("shard")
               .agg(F.count("*").alias("n_tiles"))
               .orderBy("shard")
)
display(shard_counts)
n_shards = shard_counts.count()
print(f"... {n_shards} shards at shard_z={SHARD_Z} (SF AOI)")

## 3. One PMTiles archive per shard

`gbx_pmtiles_agg(mvt_bytes, z, x, y)` is a grouped aggregate. By grouping on `shard`
instead of a single constant, each group produces its own bounded PMTiles v3 archive —
one row per shard, each containing only the tiles assigned to that parent tile.

Within each archive, multiple features that share a `(z, x, y)` tile-id are **merged**
by the aggregator (MVT blobs are decoded, features unioned per layer, and re-encoded),
so every building in every tile is preserved.

In [ ]:
# gbx_pmtiles_agg signature: gbx_pmtiles_agg(bytes, z, x, y [, metadata_json])
# Grouping by 'shard' produces one PMTiles archive (BINARY) per shard key.
shard_archives = (
    mvt_sharded
    .groupBy("shard")
    .agg(F.expr("gbx_pmtiles_agg(mvt_bytes, z, x, y)").alias("pmtiles"))
    .orderBy("shard")
)

# Collect to driver — shard count is small (4 for SF) and each archive fits in memory.
# At continental scale (thousands of shards / GB each) you would write distributed
# via a foreachPartition or a separate Spark job per shard.
shard_rows = shard_archives.collect()
print(f"... collected {len(shard_rows)} shard archives")
for r in shard_rows:
    print(f"    shard={r['shard']}  size={len(r['pmtiles']):,} bytes")

## 4. Write each shard archive to the Volume

Each archive is written as `{SHARD_DIR}/{shard}.pmtiles`. The write is FUSE-safe
sequential I/O from the driver — appropriate for a handful of shards. At large scale
you would distribute the writes: one `foreachPartition` block per Spark partition, or
a separate `spark.write` step that materialises each archive into object storage in
parallel.

In [ ]:
SHARD_DIR = f"{HELIOS_DIR}/buildings_shards"
os.makedirs(SHARD_DIR, exist_ok=True)   # FUSE-safe (idempotent on Volumes)

shard_paths = {}
for r in shard_rows:
    shard_id = r["shard"]
    path = f"{SHARD_DIR}/{shard_id}.pmtiles"
    with open(path, "wb") as fh:
        fh.write(r["pmtiles"])
    shard_paths[shard_id] = path
    print(f"... wrote {path} ({os.path.getsize(path):,} bytes)")

print(f"\n... {len(shard_paths)} shard archives in {SHARD_DIR}")

## 5. Shard catalog (Delta) + mosaic.json

For each shard we inspect the PMTiles header with `pmtiles_info` to read its bounds and
zoom range, then materialize two catalog artifacts:

1. **`sf_building_shards` (Delta table)** — one row per shard with spatial bounds,
   zoom range, and Volume path. Queryable with SQL for spatial predicates
   ("which shards intersect this viewport?").

2. **`mosaic.json` (Volume)** — a compact JSON manifest keyed by `shard`. A
   MapLibre + pmtiles.js frontend reads this to discover which archives exist and their
   bounds, then fetches the correct archive for the current viewport.

The mosaic JSON follows a simplified TileJSON-style schema:
```json
{
  "type": "mosaic",
  "shards": [
    { "id": "11_326_791", "path": "...", "bounds": [...], "min_zoom": 11, "max_zoom": 16 },
    ...
  ]
}
```

In [ ]:
from databricks.labs.gbx.pmtiles import pmtiles_info as _pmtiles_info

catalog_rows = []
mosaic_shards = []

for shard_id, path in sorted(shard_paths.items()):
    info = _pmtiles_info(path)
    bounds = info.get("bounds", [None, None, None, None])  # [min_lon, min_lat, max_lon, max_lat]
    min_zoom = info.get("min_zoom")
    max_zoom = info.get("max_zoom")

    catalog_rows.append({
        "shard": shard_id,
        "path": path,
        "min_lon": float(bounds[0]) if bounds[0] is not None else None,
        "min_lat": float(bounds[1]) if bounds[1] is not None else None,
        "max_lon": float(bounds[2]) if bounds[2] is not None else None,
        "max_lat": float(bounds[3]) if bounds[3] is not None else None,
        "min_zoom": int(min_zoom) if min_zoom is not None else None,
        "max_zoom": int(max_zoom) if max_zoom is not None else None,
    })
    mosaic_shards.append({
        "id": shard_id,
        "path": path,
        "bounds": bounds,
        "min_zoom": min_zoom,
        "max_zoom": max_zoom,
    })
    print(f"... {shard_id}: bounds={bounds} zoom={min_zoom}-{max_zoom}")

# -- Delta catalog -------------------------------------------------------------
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

_cat_schema = StructType([
    StructField("shard",    StringType(),  False),
    StructField("path",     StringType(),  True),
    StructField("min_lon",  DoubleType(),  True),
    StructField("min_lat",  DoubleType(),  True),
    StructField("max_lon",  DoubleType(),  True),
    StructField("max_lat",  DoubleType(),  True),
    StructField("min_zoom", IntegerType(), True),
    StructField("max_zoom", IntegerType(), True),
])

catalog_df = spark.createDataFrame(catalog_rows, schema=_cat_schema)
sf_shard_catalog = finalize_delta(catalog_df, "sf_building_shards")
display(sf_shard_catalog)

# -- mosaic.json ---------------------------------------------------------------
MOSAIC_PATH = f"{HELIOS_DIR}/mosaic.json"
mosaic_doc = {"type": "mosaic", "version": "1", "shards": mosaic_shards}
with open(MOSAIC_PATH, "w") as fh:
    json.dump(mosaic_doc, fh, indent=2)
print(f"... wrote {MOSAIC_PATH}")

## 6. View + explain the mosaic

`show_pmtiles` renders a single shard — it renders one archive at a time. The complete
multi-shard map is assembled **client-side**: a MapLibre + pmtiles.js client reads
`mosaic.json` to discover the archives, computes which shard(s) the current viewport
intersects, and fetches only those archives via HTTP range-requests (no tile-proxy
server required).

Two key rules for a correct mosaic:

- **No output overlap.** The same `(z, x, y)` tile must appear in at most one archive.
  The bit-shift shard assignment guarantees this — `shiftright` maps each tile to a
  unique parent.
- **Buffer the source, not the output.** If you need features that straddle a shard
  boundary to render correctly on both sides, expand the source geometry query by a
  small buffer before tiling — but let each shard's MVT encoder clip features to its
  tile boundary. The output tiles themselves remain non-overlapping.

In [ ]:
# Print header info for all shards side-by-side so we can confirm they are
# disjoint (non-overlapping bounds) and bounded (bounds != global extent).
print("Shard header summary:")
for shard_id, path in sorted(shard_paths.items()):
    info = _pmtiles_info(path)
    print(f"  {shard_id:20s}  type={info.get('tile_type')} "
          f"zoom={info.get('min_zoom')}-{info.get('max_zoom')} "
          f"bounds={info.get('bounds')}")

print()
print(f"mosaic.json ({MOSAIC_PATH}):")
with open(MOSAIC_PATH) as fh:
    _mosaic = json.load(fh)
print(f"  {len(_mosaic['shards'])} shard entries")
for _s in _mosaic["shards"]:
    print(f"  {_s['id']:20s}  bounds={_s['bounds']}")

In [ ]:
# Render the first shard archive.
# show_pmtiles renders ONE archive through the INTERACTIVE_PLOTS toggle:
#   False (default) -> static image (GitHub-renderable)
#   True            -> interactive MapLibre view
_first_shard_path = sorted(shard_paths.values())[0]
_first_shard_id   = sorted(shard_paths.keys())[0]
print(f"... rendering shard: {_first_shard_id}")
show_pmtiles(_first_shard_path)

In [ ]:
# Render a second shard to visually confirm the bounded, disjoint extents.
if len(shard_paths) > 1:
    _second_shard_path = sorted(shard_paths.values())[1]
    _second_shard_id   = sorted(shard_paths.keys())[1]
    print(f"... rendering shard: {_second_shard_id}")
    show_pmtiles(_second_shard_path)
else:
    print("... only one shard present (re-run with a wider AOI to see multiple)")

## What we built

| Artifact | Location | Description |
|---|---|---|
| `sf_buildings_mvt_shards_src` (Delta) | managed table | `(z, x, y, mvt_bytes, shard)` pyramid rows, z11–z16 |
| `sf_building_shards` (Delta) | managed table | shard catalog: bounds + zoom range + Volume path |
| `{shard_id}.pmtiles` (Volume) | `HELIOS_DIR/buildings_shards/` | one bounded PMTiles archive per shard |
| `mosaic.json` (Volume) | `HELIOS_DIR/` | TileJSON-style manifest; maps shard bounds → archive path |

**When to use sharding vs a single archive:**

- **Single archive** (Notebook 01) — city-scale data that fits comfortably in one
  file (< ~2 GB); fastest to generate and serve.
- **Distributed sharding** (this notebook) — statewide, national, or global data;
  when any sub-region updates independently; when parallel workers must avoid write
  contention; when you want atomic per-region re-generation without touching the rest
  of the dataset.

The `mosaic.json` is the bridge: a lightweight catalog that lets the client assemble
the full multi-region map from the correct subset of archives for any viewport —
no tile-proxy server, no merge step, no global re-write when a neighbourhood changes.

**Next:** Notebook 02 drapes NAIP aerial imagery behind these footprints as a visual
basemap; Notebook 03 adds solar-yield terrain scoring.